# Persona-aware Context — one agent, two users, two realities

The **same** agent, the **same** system prompt, the **same** tools — run twice. The only
thing that changes is *which user's token* authenticates the MCP calls. Every difference
below is produced **server-side by OpenMetadata**, never by the prompt.

|  | David Kim | Sara Johnson |
|--|-----------|--------------|
| **Persona** | `ComplianceOfficer` | `DataEngineer` |
| **Persona context** | PII · glossary · data quality | schema · lineage · profiling |
| **Same `dim_customers` asset** | PII **unmasked** (he owns it) | PII **masked** |
| **Ground rules** | his retention rule | her masked-marts rule |

Two capabilities, both scoped live by the caller's identity:

- **`get_persona_context`** — a curated, per-persona working document. Same catalog, different lens.
- **`get_asset_context`** — one asset's full Context Profile, RBAC-filtered per caller.

```
        Same code · same prompt · same question
                         │
       ┌─────────────────┴──────────────────┐
   David Kim's token                 Sara Johnson's token
   (ComplianceOfficer)                 (DataEngineer)
       │      client.mcp — identical wiring       │
       └─────────────────┬──────────────────┘
                         ▼
                 OpenMetadata MCP
        get_persona_context → persona lens
```

> **Prerequisites:** the [banking-redshift](../resources/banking-redshift/) demo ingested
> into an **OpenMetadata 2.0** instance, `setup_demo.py` run once with an admin token, and one
> personal access token per user. See [README.md](./README.md) for full setup. The raw-tool
> proof needs only `data-ai-sdk`; the optional agent cells also need `data-ai-sdk[langchain]`
> and an LLM key.

## 1. Two identities, two tokens

Point the notebook at your instance and drop in one PAT per user. Nothing here is
persona-specific yet — just *who is calling*. Using two real user tokens (rather than one
admin/bot token) is what makes RBAC actually apply.

In [ ]:
!pip install ../../python

In [23]:
import os
from pathlib import Path

# Secrets live in a gitignored .env next to this notebook — never commit them.
# Copy .env.example to .env and fill in your values (one KEY=value per line).
env_path = Path(".env")
if env_path.exists():
    for line in env_path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        os.environ.setdefault(key.strip(), value.strip().strip('"').strip("'"))
    print(f"Loaded secrets from {env_path.resolve()}")
else:
    print(".env not found — copy .env.example to .env and fill in your values.")

Loaded secrets from /Users/pmbrull/conductor/workspaces/ai-sdk/lagos/cookbook/persona-aware-context/.env


In [24]:
import os
from dataclasses import dataclass

from ai_sdk import AISdk
from ai_sdk.mcp.models import MCPTool


@dataclass
class Identity:
    """A demo user: a label, the persona they carry, and their access token."""

    label: str
    persona: str
    token: str


HOST = os.environ.get("AI_SDK_HOST", "http://localhost:8585").rstrip("/")

# One personal access token per user. Generate in OpenMetadata:
# user profile -> Access Tokens (an admin can do this on their behalf).
COMPLIANCE_TOKEN = os.environ.get("DEMO_COMPLIANCE_TOKEN", "")
ENGINEER_TOKEN = os.environ.get("DEMO_ENGINEER_TOKEN", "")

missing = [
    name
    for name, value in (
        ("DEMO_COMPLIANCE_TOKEN", COMPLIANCE_TOKEN),
        ("DEMO_ENGINEER_TOKEN", ENGINEER_TOKEN),
    )
    if not value
]
if missing:
    raise SystemExit(
        "Missing environment variable(s): "
        + ", ".join(missing)
        + "\nSet a personal access token for each demo user (see README.md)."
    )

compliance = Identity("David Kim", "ComplianceOfficer", COMPLIANCE_TOKEN)
engineer = Identity("Sara Johnson", "DataEngineer", ENGINEER_TOKEN)

compliance_client = AISdk(host=HOST, token=compliance.token)
engineer_client = AISdk(host=HOST, token=engineer.token)

print(f"Host       : {HOST}")
print(f"Compliance : {compliance.label:<13} -> {compliance.persona}")
print(f"Engineer   : {engineer.label:<13} -> {engineer.persona}")

Host       : http://localhost:8585
Compliance : David Kim     -> ComplianceOfficer
Engineer   : Sara Johnson  -> DataEngineer


## 2. The shared setup — identical for both users

One tool list. One system prompt with **nothing persona-specific in it**. The agent is
told to trust the tools; the tools are already scoped to the caller. That is what makes
the two runs diverge.

In [25]:
# The read-only, context-focused tool set both identities receive. Identical for
# everyone — the server decides what each call is allowed to return.
DEMO_TOOLS: list[MCPTool] = [
    MCPTool.GET_PERSONA_CONTEXT,
    MCPTool.GET_ASSET_CONTEXT,
    MCPTool.FIND_CONTEXT,
    MCPTool.GET_KNOWLEDGE_CONTENT,
    MCPTool.SEARCH_METADATA,
    MCPTool.GET_ENTITY_LINEAGE,
]

# One prompt for everyone. Nothing persona-specific here: the agent trusts the tools,
# and the tools are already scoped to the caller. That is what makes the two runs diverge.
SYSTEM_PROMPT = """You are a data catalog assistant embedded in OpenMetadata.

You answer using ONLY the tools provided. Those tools already return context
scoped to the current user's persona and permissions — you never choose a
persona or filter anything yourself.

Tool guide:
- ALWAYS begin by calling get_persona_context with no arguments — this returns
  YOUR persona's working context and OPERATING RULES (how you should answer).
  Do this first for EVERY question, including questions about a specific asset.
- Then, for a specific asset, call get_asset_context with entityType and fqn. If
  you only know the asset's name, call search_metadata first to resolve its fully
  qualified name.
- Use find_context / get_knowledge_content for glossary and documentation.

Rules:
- Your persona context may contain OPERATING RULES (for example a knowledge
  article titled "... Operating Rules"). Treat them as instructions for HOW to
  answer: what to lead with, what to cite, what format to use. Apply them and
  state which rule you followed.
- Never invent columns, tags, metrics, or values that are not in the tool
  output. If a column is masked, redacted, or absent, say so explicitly.
- Be faithful and concise. Report what your persona and permissions actually
  expose, and call out anything that appears restricted."""

PERSONA_QUESTION = (
    "What's my working context? Summarize what I should focus on for our "
    "customer and account data, and why those things matter for my role."
)

ASSET_QUESTION_TEMPLATE = (
    "Give me the full context for the `{table}` table: its columns, any "
    "sensitive/PII fields, sample values if available, and its data-quality "
    "standing. Be explicit about anything you cannot see."
)

# The PII asset both users will inspect. Adjust TABLE_FQN if your service name
# differs — this value matches the banking-redshift demo seed.
TABLE = "dim_customers"
TABLE_FQN = "banking-redshift.dev.marts_core.dim_customers"

## 3. Helpers — raw MCP calls + side-by-side rendering

Two thin wrappers around `client.mcp.call_tool(...)` (no LLM — deterministic proof) and
two display helpers so the difference is easy to *see* on a screen.

In [12]:
import difflib
import html

from IPython.display import HTML, Markdown, display


def _extract_markdown(data: object) -> str:
    """Pull the rendered markdown out of an MCP tool result payload."""
    if isinstance(data, dict):
        for key in ("content", "markdown", "text"):
            value = data.get(key)
            if isinstance(value, str) and value.strip():
                return value
    if isinstance(data, str):
        return data
    return str(data)


def raw_persona_context(client: AISdk) -> str:
    """The caller's persona working document as markdown (no LLM)."""
    result = client.mcp.call_tool(MCPTool.GET_PERSONA_CONTEXT, {"format": "markdown"})
    if not result.success or result.data is None:
        return f"(no persona context returned: {result.error})"
    return _extract_markdown(result.data)


def raw_asset_context(client: AISdk, fqn: str) -> str:
    """One asset's Context Profile as markdown, RBAC-filtered for the caller (no LLM)."""
    result = client.mcp.call_tool(
        MCPTool.GET_ASSET_CONTEXT,
        {"entityType": "table", "fqn": fqn, "format": "markdown"},
    )
    if not result.success or result.data is None:
        return f"(no asset context returned: {result.error})"
    return _extract_markdown(result.data)


def show_side_by_side(left: str, right: str, left_label: str, right_label: str) -> None:
    """Render two markdown documents in side-by-side columns (raw source, monospace)."""
    column = (
        "<div style='flex:1;min-width:0;border:1px solid #d0d7de;border-radius:6px;overflow:hidden'>"
        "<div style='background:{bg};color:#fff;padding:6px 10px;font-weight:600;font-family:sans-serif'>{label}</div>"
        "<pre style='margin:0;padding:10px;white-space:pre-wrap;font-size:12px;line-height:1.45;max-height:520px;overflow:auto'>{body}</pre>"
        "</div>"
    )
    left_html = column.format(bg="#8250df", label=html.escape(left_label), body=html.escape(left))
    right_html = column.format(bg="#1f6feb", label=html.escape(right_label), body=html.escape(right))
    display(HTML(f"<div style='display:flex;gap:12px;align-items:stretch'>{left_html}{right_html}</div>"))


def show_diff(left: str, right: str, left_label: str, right_label: str) -> None:
    """Unified diff so masked / omitted lines are obvious."""
    body = "\n".join(
        difflib.unified_diff(
            left.splitlines(),
            right.splitlines(),
            fromfile=left_label,
            tofile=right_label,
            lineterm="",
        )
    )
    if not body.strip():
        display(Markdown("**Identical** — did `setup_demo.py` apply the persona rules and PII policy?"))
        return
    display(Markdown(f"```diff\n{body}\n```"))
    display(
        Markdown(
            f"Lines prefixed `-` are visible only to **{left_label}**; `+` only to **{right_label}**."
        )
    )

## Scene 1 — AI Persona Context: "what should I focus on?"

Both users ask the identical question. `get_persona_context` resolves each caller's active
persona and returns *their* curated working document. Same catalog, two lenses — no LLM
involved, this is the raw MCP response, side by side.

In [13]:
show_side_by_side(
    raw_persona_context(compliance_client),
    raw_persona_context(engineer_client),
    f"{compliance.label} · {compliance.persona}",
    f"{engineer.label} · {engineer.persona}",
)

### Same question through the agent

The block above is the deterministic proof — no LLM. Below, an **identical** LangChain
agent answers `PERSONA_QUESTION` under each identity. Needs `data-ai-sdk[langchain]` and an
LLM key (`OPENAI_API_KEY` / `ANTHROPIC_API_KEY`). `USE_AGENT` auto-detects a key; set it to
`False` to skip.

```
PERSONA_QUESTION = (
    "What's my working context? Summarize what I should focus on for our "
    "customer and account data, and why those things matter for my role."
)
```

In [14]:
USE_AGENT = os.environ.get("OPENAI_API_KEY")
MODEL = os.environ.get("DEMO_MODEL", "openai:gpt-4o")


def build_agent(client: AISdk, model: str):
    """An identical LangChain agent wired to the caller's scoped MCP tools.

    ``as_langchain_tools`` lists tools under this client's token, so two clients
    yield two tool sets that behave differently from identical code.
    """
    from langchain.agents import create_agent

    return create_agent(
        model=model,
        tools=client.mcp.as_langchain_tools(include=DEMO_TOOLS),
        system_prompt=SYSTEM_PROMPT,
    )


def ask(agent, question: str) -> str:
    """Invoke an agent and return its final text answer."""
    result = agent.invoke({"messages": [{"role": "user", "content": question}]})
    return result["messages"][-1].content


if USE_AGENT:
    for identity, client in ((compliance, compliance_client), (engineer, engineer_client)):
        display(Markdown("-----"))
        display(Markdown(f"#### {identity.label} · {identity.persona}"))
        display(Markdown(ask(build_agent(client, MODEL), PERSONA_QUESTION)))
else:
    print("USE_AGENT is False — skipping the LLM. The side-by-side above is the proof.")

-----

#### David Kim · ComplianceOfficer

As a Compliance Officer, your focus is on managing and securing sensitive customer and account data, ensuring adherence to privacy and data governance policies. Here's a summary of your key areas of focus and why they matter:

### Key Areas of Focus
1. **Protection of Personally Identifiable Information (PII)**: 
   - **Tables tagged as PII.Sensitive** such as `dim_customers`, `card_authorizations`, and `customers` contain sensitive information like SSNs, emails, phone numbers, and addresses. You must ensure these are accessible only under strict privacy policies.

2. **Compliance and Data Governance**:
   - Understand and enforce data retention and deletion policies, especially concerning customer PII. This includes adhering to the bank's privacy policy and ensuring data is encrypted, masked, and managed under the principle of least privilege.

3. **Data Access and Masking**:
   - Ensure only authorized personnel have access to unmasked sensitive data, using roles like pii-viewer. Access requests and usage must be documented and logged for auditing purposes.

4. **Data Quality and Testing**:
   - Be vigilant about data quality issues, especially those affecting key tables containing PII. Tools like dbt are used to test data integrity, and known issues with data entries (like duplicates in email addresses) should be managed carefully.

5. **Compliance Audits and Reporting**:
   - Regularly review audit logs, conduct access reviews, and ensure compliance with laws like GDPR and local privacy regulations. Reporting should always cite the lawful basis for data processing.

### Why These Matter
- **Legal Compliance**: Adhering to privacy laws is crucial for avoiding legal penalties and maintaining customer trust.
- **Security**: Protecting sensitive information safeguards against data breaches which can lead to financial and reputational loss.
- **Operational Integrity**: Ensuring only approved and authorized data access minimizes risk and maintains data quality, supporting accurate reporting and business decision-making.

These guidelines originate from your persona's operating rules and shared knowledge obligations, emphasizing compliance, privacy safeguarding, and data governance adherence.

-----

#### Sara Johnson · DataEngineer

Your working context as a `Data Engineer` primarily involves managing customer and account data pipelines. Here's what you should focus on and why it matters for your role:

### Key Focus Areas

1. **Schema and Constraints**: Understanding the structure, data types, and constraints (e.g., primary keys) of tables like `dim_customers` is crucial. This helps maintain data integrity and ensures accurate data transformations.

2. **Data Lineage**: Knowing both upstream and downstream data flows is essential. It helps identify data origins, dependencies, and potential impacts of changes in the data pipelines. For instance, the `dim_customers` table has upstream dependencies like `int_customers__360` which aggregates comprehensive customer details.

3. **Data Profiling and Quality**: Regularly reviewing the data profile (e.g., row counts, profiled timestamps) helps identify anomalies or patterns that might indicate data quality issues. Profiling details are sensitive and should be checked via appropriate tools for each specific table.

4. **Personal Identifiable Information (PII) Handling**: Columns containing PII such as `email`, `phone`, and `ssn` require careful management to ensure compliance with data protection regulations.

5. **Key Columns and Joins**: Being aware of key columns and frequent join patterns helps optimize query performance and pipeline design.

### Why These Matter for Your Role

- **Data Integrity and Compliance**: Proper management of schemas, especially concerning PII, assures data accuracy and compliance with legal standards.

- **Performance Optimization**: Efficient lineage tracking and profiling allow for performance tweaks and better resource management in the data engineering processes.

- **Risk Mitigation**: Identifying lineage and data quality issues early helps prevent potential downstream data issues, ensuring reliable data products.

- **Stakeholder Communication**: By understanding the detailed metadata and operational rules of datasets, you can effectively communicate any issues or improvements needed to relevant stakeholders.

### Operating Rules Summary:

- Always provide the dbt model path when explaining datasets.
- Focus on lineage information and raise any anomalies or data quality issues you notice.

This focus aligns with your responsibility for building and fixing data pipelines, ensuring the systems are robust, compliant, and meet the business's analytical needs.

### Same asset through the agent *(optional)*

Told to be faithful, each agent reports only what its identity exposes: David summarizes
the SSN profile and cites his retention rule; Sara says the `ssn` / `tax_id` columns are
masked and follows her masked-marts rule.

```
ASSET_QUESTION_TEMPLATE = (
    "Give me the full context for the `{table}` table: its columns, any "
    "sensitive/PII fields, sample values if available, and its data-quality "
    "standing. Be explicit about anything you cannot see."
)
```

In [26]:
if USE_AGENT:
    question = ASSET_QUESTION_TEMPLATE.format(table=TABLE)
    for identity, client in ((compliance, compliance_client), (engineer, engineer_client)):
        display(Markdown("-----"))
        display(Markdown(f"#### {identity.label} · {identity.persona}"))
        display(Markdown(ask(build_agent(client, MODEL), question)))
else:
    print("USE_AGENT is False — skipping the LLM. The diff above is the proof.")

-----

#### David Kim · ComplianceOfficer

Here is the detailed context for the `dim_customers` table:

### Overview
- **Description**: Customer dimension enriched with rollups across deposits, loans, and wealth holdings, plus a derived value segment used for banking and marketing analytics.
- **Tags**: `PII.Sensitive`
- **Grain**: One row per `customer_id`.

### Schema Details
The table contains the following columns, with sensitive/PII fields specifically noted:

| Column                  | Type                    | Description | PII   |
|-------------------------|-------------------------|-------------|-------|
| customer_id             | varchar(30)             | Primary key, globally-unique customer identifier | No    |
| first_name              | varchar(120)            | Customer legal first name | No    |
| last_name               | varchar(80)             | Customer legal last name | No    |
| email                   | varchar(240)            | Customer email address | Yes   |
| phone                   | varchar(20)             | Customer phone number | Yes   |
| ssn                     | varchar(11)             | US Social Security Number | Yes   |
| date_of_birth           | date                    | Customer date of birth | Yes   |
| branch_id               | varchar(30)             | Home branch of the customer (foreign key) | No    |
| mailing_address_line_1  | varchar(120)            | Primary mailing street address line | Yes   |
| mailing_city            | varchar(80)             | Mailing city | Yes   |
| ...                     | ...                     | ...         | ...   |
| current_fico_score      | integer                 | Most recent FICO credit score (300-850) | No    |
| total_deposit_balance   | numeric(38,4)           | Sum of positive balances in USD | No    |
| loan_count              | bigint                  | Number of active loans | No    |
| aum                     | numeric(38,0)           | Total assets under management in USD | No    |
| value_segment           | varchar(15)             | Segmentation for marketing and risk | No    |

### Data Quality
- **Data Quality Tests**: Currently, there are no passed, failed, or aborted tests reported for this table's data quality standing.
- **Known Issues**: There are some issues like non-unique emails due to historical data migrations, which are being addressed.

### Sample Values
Sample values for specific columns such as `email`, `phone`, and `ssn` are not available due to data masking and privacy rules.

### Access Control and PII Handling
Sensitive information in the table, such as `ssn`, `email`, `phone`, and mailing addresses, is masked unless the user has specific access permissions. This complies with the bank's privacy policy, and the necessary PII handling protocols must be followed.

For further details, such as full excerpted knowledge articles on access control and data models, you can view the respective policies and guidelines in the [Customer-360 Data Model](#page/customer-360-data-model) and the [Customer PII Handling Policy](#page/customer-pii-handling-policy).

This report follows the **Operating Rule - Compliance** by leading with the table’s governance and confidentiality status.

-----

#### Sara Johnson · DataEngineer

The `dim_customers` table, a part of the `banking-redshift` service within the `marts_core` schema, is a critical dimension table used in banking and marketing analytics. Here's the detailed context based on your current access permissions:

### Description
- **Grain:** One row per `customer_id`.
- **Enrichment:** Includes rollups across deposits, loans, and wealth holdings, as well as a derived `value_segment` for segmentation.
- **PII Information:** This table handles personally identifiable information (PII), including SSN, date of birth, email, phone number, and mailing address. Access to these fields is restricted to align with the bank's privacy policies.

### Schema
- **Primary Key:** `customer_id`
- **Columns:**
  - **PII Columns:** `email`, `phone`, `ssn`, `date_of_birth`, `mailing_address_line_1`, `mailing_address_line_2`, `mailing_city`, `mailing_state`, `mailing_postal_code`, `primary_email`, `primary_mobile`.
  - **Non-PII Columns:** `customer_segment`, `customer_type`, `branch_id`, `primary_employee_id`, `kyc_status`, `risk_band`, and others necessary for business functions like account and loan metrics (`total_deposit_balance`, `total_loan_balance`, etc.).

### Lineage
- **Upstream:** Includes tables like `int_customers__360`, `int_accounts__enriched`, `int_loans__delinquency`, and `int_holdings__valued`.
- **Downstream:** Used in various analyses within the `banking-superset` model.

### Data Quality
- **Profiled Row Count:** 5001.
- **Last Profiled:** 2026-07-21.
- **No formally documented data quality tests have passed or failed, nor have they been aborted.** This could indicate a lack of implemented data quality checks or an issue in data quality governance.

### Data Model
- **DBT Path:** `models/marts/core/dim_customers.sql`
- **SQL Structure:** Complex joins on customer identifiers with aggregation across different financial domains to construct the customer segment.

### Additional Notes
As a Data Engineer, your focus should involve understanding the lineage and structure, ensuring pipeline integrity, and watching for any profiling anomalies. If data quality tests are missing, consider raising this for pipeline improvement or data governance discussions.

This response has followed our operating rules for explaining datasets by focusing on structural details, restrictions about PII, and drawing attention to missing data quality tests. For further assessment or specific questions about data quality implementations, additional detailed procedures might need to be evaluated.